In [1]:
# Imports

In [2]:
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import json
import os
from tqdm import tqdm
import time
import shutil
from datetime import datetime

In [3]:
# Title: Initialize Spotify API Client with Credentials

# Description:
# Authenticates and initializes a Spotipy client using Spotify Client Credentials flow, 
# allowing access to public Spotify Web API endpoints such as artist, track, and genre data.

In [4]:
# Insert personal client id and client secret
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id='bcfc6207ac75490896635160936ed618',
    client_secret='7d3fa689cfbf4c46b96a1553340c82ab'
))

In [5]:
# Title: Fetch and Rank Top Genres per Year with Live Spotify API Lookup

# Description:
# Creates a save directory and loads cached artist-to-genre mappings, 
# defines Wrapped cutoff dates from 2017 to 2024, 
# filters top 200 songs per year based on play count, 
# attempts to fetch missing genre data from the Spotify API with caching and error handling, 
# collects and counts genre occurrences among valid artists, 
# prints the top 5 genres per year, and 
# saves the updated genre map to a local JSON cache for future use.

In [6]:
# Create the directory to save data
SAVE_DIR = 'wrapped_data'
os.makedirs(SAVE_DIR, exist_ok=True)

# Load artist genre cache, due to limited API calls
CACHE_PATH = os.path.join(SAVE_DIR, 'top_genres.json')
artist_genre_map = {}

if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, 'r') as f:
        artist_genre_map = json.load(f)

# Load streaming data
streaming_data = pd.read_csv(os.path.join(SAVE_DIR, 'streaming_data.csv'))
streaming_data['ts'] = pd.to_datetime(streaming_data['ts'])

# Wrapped end dates per year (include 2025!)
wrapped_end_dates = {
    year: f"{year}-{'10-31' if year <= 2019 else '11-15'}T23:59:59Z"
    for year in range(2017, 2026)
}

# Main analysis
top_genres_by_year = {}

for year, end_str in wrapped_end_dates.items():
    print(f"\nWrapped {year} – Finding Top Genres")

    # Filter for current year range
    start = f"{year}-01-01T00:00:00Z"
    year_data = streaming_data.query("@start <= ts <= @end", local_dict={'start': start, 'end': end_str})

    if year_data.empty:
        print("No data for this year.")
        top_genres_by_year[year] = []
        continue

    # Get top 200 songs by play count for this year
    top_candidates = (
        year_data.groupby(['master_metadata_track_name', 'master_metadata_album_artist_name'])
        .agg(play_count=('ts', 'count'))
        .reset_index()
        .sort_values('play_count', ascending=False)
        .head(200)
    )

    valid_artists, genres = [], []

    for artist in tqdm(top_candidates['master_metadata_album_artist_name'].dropna().unique(), desc=f"Fetching genres for {year}"):
        artist_genres = artist_genre_map.get(artist)

        if not artist_genres:
            try:
                results = sp.search(q=f"artist:{artist}", type='artist', limit=1)
                artist_genres = results.get('artists', {}).get('items', [{}])[0].get('genres', [])
                artist_genre_map[artist] = artist_genres if artist_genres else ['Unknown']
                time.sleep(0.2)
            except Exception as e:
                print(f"Error fetching {artist}: {e}")
                artist_genre_map[artist] = ['Unknown']

            artist_genres = artist_genre_map[artist]

        if artist_genres and 'Unknown' not in artist_genres:
            valid_artists.append(artist)
            genres += artist_genres

        if len(valid_artists) >= 100:
            break

    if not genres:
        print(f"No valid genres found for {year}.")
        top_genres_by_year[year] = []
        continue

    # Store top 5 genres
    top_genres_by_year[year] = pd.Series(genres).value_counts().head(5)

    print(f"Top Genres for {year}:")
    print(top_genres_by_year[year])

# Save updated cache
with open(CACHE_PATH, 'w') as f:
    json.dump(artist_genre_map, f, indent=2)

print(f"\nArtist genres saved to {CACHE_PATH}.")



Wrapped 2017 – Finding Top Genres


Fetching genres for 2017: 100%|█████████████| 43/43 [00:00<00:00, 371100.97it/s]


Top Genres for 2017:
indie              7
surf rock          4
garage rock        4
psychedelic pop    3
indie rock         3
dtype: int64

Wrapped 2018 – Finding Top Genres


Fetching genres for 2018: 100%|████████████| 69/69 [00:00<00:00, 1252844.05it/s]


Top Genres for 2018:
indie           7
garage rock     4
surf rock       3
classic rock    3
baroque pop     3
dtype: int64

Wrapped 2019 – Finding Top Genres


...

Top Genres for 2023:
indie              7
bedroom pop        6
alternative r&b    5
baroque pop        5
chamber pop        4
dtype: int64

Wrapped 2024 – Finding Top Genres


Fetching genres for 2024: 100%|██████████| 102/102 [00:00<00:00, 1590405.23it/s]


Top Genres for 2024:
musicals       9
indie          8
retro soul     5
garage rock    5
indie folk     5
dtype: int64

Wrapped 2025 – Finding Top Genres


Fetching genres for 2025: 100%|██████████| 118/118 [00:00<00:00, 1256162.11it/s]

Top Genres for 2025:
musicals       12
bedroom pop     5
indie           4
motown          4
soul            4
dtype: int64

Artist genres saved to wrapped_data/top_genres.json.


In [10]:
# Title: Filter and Backup Cleaned Artist Genre Map

# Description:
# Loads a cached artist-to-genre dictionary, 
# removes entries where genre is missing or marked as 'Unknown', 
# creates a timestamped backup of the existing clean genre file if present, 
# saves the cleaned dictionary to a new JSON file, 
# and prints a summary report of how many artists were retained or dropped.

In [11]:
# Set up paths
SAVE_DIR = 'wrapped_data'
os.makedirs(SAVE_DIR, exist_ok=True)

CACHE_PATH = os.path.join(SAVE_DIR, 'top_genres.json')
CLEAN_CACHE_PATH = os.path.join(SAVE_DIR, 'top_genres_clean.json')

# Load your current genre map
with open(CACHE_PATH, 'r') as f:
    artist_genre_map = json.load(f)

# Clean the genre map
clean_artist_genre_map = {}

for artist, genres in artist_genre_map.items():
    if genres and isinstance(genres, list) and 'Unknown' not in genres:
        clean_artist_genre_map[artist] = genres

# Backup old clean file if it exists 
if os.path.exists(CLEAN_CACHE_PATH):
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    backup_path = f"{CLEAN_CACHE_PATH}.{timestamp}.bak"
    shutil.copy(CLEAN_CACHE_PATH, backup_path)
    print(f"Backup created: {backup_path}")

# Save the clean genre map into a new file
with open(CLEAN_CACHE_PATH, 'w') as f:
    json.dump(clean_artist_genre_map, f, indent=2)

# Print a report
original_count = len(artist_genre_map)
clean_count = len(clean_artist_genre_map)
dropped_count = original_count - clean_count

print(f"""
Clean Report:
- Original artist entries: {original_count}
- Artists with real genres kept: {clean_count}
- Artists dropped (only 'Unknown'): {dropped_count}

Clean genre map saved as: {CLEAN_CACHE_PATH}
""")

Backup created: wrapped_data/top_genres_clean.json.20250710_161844.bak

Clean Report:
- Original artist entries: 583
- Artists with real genres kept: 310
- Artists dropped (only 'Unknown'): 273

Clean genre map saved as: wrapped_data/top_genres_clean.json

